# Window classifier — training (Colab / Kaggle)

1. Локально сгенерируйте CSV: `python prepare_dataset.py --from YYYY-MM-DD --to YYYY-MM-DD --out data/raw_windows.csv` (из каталога `ml/`).
2. Загрузите файл сюда (Colab: следующая ячейка) или укажите путь в Kaggle Input.
3. Запустите ячейки по порядку. Артефакты: `classifier.joblib`, `training_metadata.json`, `metrics.txt`.

Разбиение по `session_id`, GPU-колонки при `DROP_GPU_MODE="auto"` отбрасываются, если везде нули; текстовые фичи one-hot.

In [ ]:
%pip install -q pandas scikit-learn numpy joblib

### Загрузка CSV (Colab)
Раскомментируйте и выполните один раз, если файл не в среде выполнения:

In [ ]:
# from google.colab import files
# uploaded = files.upload()   # выберите raw_windows.csv
# CSV_PATH = list(uploaded.keys())[0]

# Kaggle / локальный путь к экспортированному датасету:
CSV_PATH = "raw_windows.csv"

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- Синхронно с ml/constants.py ---
LABEL_COL = "label"
GPU_COLS = ("gpu_util_avg", "gpu_temp_avg", "gpu_mem_avg_mb")
META_COLS = (
    "user_id", "session_id", "window_index", "window_start", "window_end", "label"
)
TEXT_COLS = ("active_process", "foreground_window_title", "game_name")

SEED = 42
DROP_GPU_MODE = "auto"  # "auto" | "always" | "never"
VAL_FRAC = 0.15
TEST_FRAC = 0.15
OUT_DIR = Path("training_out")


def numeric_feature_columns(df: pd.DataFrame, drop_gpu: set[str]) -> list[str]:
    skip = set(META_COLS) | set(TEXT_COLS) | drop_gpu
    cols = []
    for c in df.columns:
        if c in skip or c == LABEL_COL:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return sorted(cols)


df = pd.read_csv(CSV_PATH)
df = df[df[LABEL_COL].astype(str).str.strip() != ""].copy()
assert len(df) >= 10, "Нужно >= 10 размеченных строк"
assert "session_id" in df.columns

drop_gpu: set[str] = set()
if DROP_GPU_MODE == "always":
    drop_gpu = set(GPU_COLS)
elif DROP_GPU_MODE == "never":
    drop_gpu = set()
else:
    for c in GPU_COLS:
        if c not in df.columns:
            drop_gpu.add(c)
            continue
        s = pd.to_numeric(df[c], errors="coerce").fillna(0)
        if np.nanmax(np.abs(s.values)) <= 1e-9:
            drop_gpu.add(c)

num_cols = numeric_feature_columns(df, drop_gpu)
text_cols = [c for c in TEXT_COLS if c in df.columns]
for c in text_cols:
    df[c] = df[c].fillna("").astype(str)

y = df[LABEL_COL].astype(str)
groups = df["session_id"].values
idx = np.arange(len(df))

gss_test = GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=SEED)
tr_val_idx, te_idx = next(gss_test.split(idx, y, groups=groups))

df_tv = df.iloc[tr_val_idx].reset_index(drop=True)
y_tv = y.iloc[tr_val_idx].reset_index(drop=True)
groups_tv = df_tv["session_id"].values
idx_tv = np.arange(len(df_tv))

rel_val = VAL_FRAC / max(1e-9, (1.0 - TEST_FRAC))
rel_val = min(0.99, max(0.01, rel_val))
gss_val = GroupShuffleSplit(n_splits=1, test_size=rel_val, random_state=SEED + 1)
try:
    tr_idx, va_idx = next(gss_val.split(idx_tv, y_tv, groups=groups_tv))
except ValueError:
    tr_idx, va_idx = idx_tv, np.array([], dtype=int)

df_train = df_tv.iloc[tr_idx]
df_val = df_tv.iloc[va_idx] if len(va_idx) else df_tv.iloc[[]]
df_test = df.iloc[te_idx]

feature_cols = num_cols + text_cols
X_train = df_train[feature_cols]
y_train = y_tv.iloc[tr_idx]
X_val = df_val[feature_cols] if len(df_val) else pd.DataFrame(columns=feature_cols)
y_val = y_tv.iloc[va_idx] if len(va_idx) else pd.Series(dtype=str)
X_test = df_test[feature_cols]
y_test = y.iloc[te_idx].reset_index(drop=True)

assert len(X_train) > 0, "Пустой train — мало сессий"

transformers = [("num", StandardScaler(), num_cols)]
if text_cols:
    transformers.append(
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", max_categories=40, sparse_output=False),
            text_cols,
        )
    )

pre = ColumnTransformer(transformers, remainder="drop", verbose_feature_names_out=False)
clf = HistGradientBoostingClassifier(
    max_depth=12, learning_rate=0.08, max_iter=200, random_state=SEED
)
pipe = Pipeline([("prep", pre), ("clf", clf)])
pipe.fit(X_train, y_train)

OUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(pipe, OUT_DIR / "classifier.joblib")

labels = sorted(y.unique().tolist())
report_lines: list[str] = []


def eval_split(name: str, Xp, yp):
    if len(Xp) == 0:
        report_lines.append(f"=== {name} (empty) ===\n")
        return
    pred = pipe.predict(Xp)
    report_lines.append(f"=== {name} (n={len(Xp)}) ===\n")
    report_lines.append(classification_report(yp, pred, zero_division=0))
    macro = f1_score(yp, pred, average="macro", zero_division=0)
    report_lines.append(f"macro-F1: {macro:.4f}\n")


eval_split("train", X_train, y_train)
if len(X_val):
    eval_split("validation", X_val, y_val)
eval_split("test", X_test, y_test)

report_txt = "".join(report_lines)
print(report_txt)
(OUT_DIR / "metrics.txt").write_text(report_txt, encoding="utf-8")

meta = {
    "csv": str(CSV_PATH),
    "seed": SEED,
    "drop_gpu_mode": DROP_GPU_MODE,
    "dropped_gpu_columns": sorted(drop_gpu),
    "numeric_features": num_cols,
    "text_features": text_cols,
    "labels": labels,
    "session_counts": {
        "train": int(df_train["session_id"].nunique()),
        "val": int(df_val["session_id"].nunique()) if len(df_val) else 0,
        "test": int(df_test["session_id"].nunique()),
    },
    "row_counts": {
        "train": int(len(X_train)),
        "val": int(len(X_val)),
        "test": int(len(X_test)),
    },
}
(OUT_DIR / "training_metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Saved:", OUT_DIR / "classifier.joblib")

### Скачать артефакты (Colab)

In [ ]:
# from google.colab import files
# files.download('training_out/classifier.joblib')
# files.download('training_out/training_metadata.json')
# files.download('training_out/metrics.txt')